# 第 14 课｜R² 与策略评估

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`14_youtube-N56u1XU5x80/index.html`  
> **所属模块**：P1-M3 · 统计证据与可预测性

## 本节量化问题

高 R² 是解释力、预测力，还是过拟合信号？

## 准确定义与策略报告位置

R² 衡量相对均值基准的样本内解释比例；样本外 R² 比样本内值更接近预测问题。

- **核心关系**：R²=1-SSE/SST；样本外 R² 使用测试集误差与测试集均值基准比较。
- **写入报告**：样本内解释、样本外预测与基准
- **P2 衔接**：P2 第 49–50 课：回归与时序机器学习。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 比较样本内和样本外 R²

按时间顺序切分，不能随机打乱。特征只使用昨天及更早的信息，目标是下一日收益。

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
market = close["SIM_MARKET"].pct_change(fill_method=None)
dataset = pd.DataFrame({
    "lag1": market.shift(1),
    "lag2": market.shift(2),
    "target": market,
}).dropna()

split = int(len(dataset) * 0.7)
train, test = dataset.iloc[:split], dataset.iloc[split:]
model = LinearRegression().fit(train[["lag1", "lag2"]], train["target"])

train_r2 = model.score(train[["lag1", "lag2"]], train["target"])
test_pred = model.predict(test[["lag1", "lag2"]])
test_r2 = r2_score(test["target"], test_pred)

display(pd.Series({"train_r2": train_r2, "test_r2": test_r2, "test_observations": len(test)}))

train_r2               0.000799
test_r2               -0.002564
test_observations    226.000000
dtype: float64

## 方法边界与阶段验收

低 R² 不一定无交易价值，高 R² 也不保证经济收益；必须结合方向准确率、成本和稳定性。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
